# Fourier expressivity of the monitored QELM

Issue #309. The QELM encodes the input once, `rho(theta) = prod_i (I + sin(theta_i) X_i + cos(theta_i) Z_i) / 2` (angle encoding), so every mean the readout can see is a combination of the `3^N` Bloch monomials `B_alpha(theta)`: a trigonometric polynomial of degree at most one in each angle. The intermediate measurements, `p` and the symmetry of the gates never add a frequency; they set the coefficients. This notebook computes them exactly, on fixed circuits (`fixed_instance`), with no sampled trajectory:

- **features**: the record `M_{t,i}` (mean `p Tr[Z_i Phi_t(rho)]`, `Phi_t` = the layers with the Z-dephasing of probability `p`) and the final `Z_i`;
- **transfer matrix** `C(p)`: rows = features, columns = monomials, `C_{k alpha} = Tr[O_k sigma_alpha] / 2^N` with `O_k` the Heisenberg image of `Z_i`;
- **measures** (`QELM.evaluation.fourier`): the rank of `C(p)` (dimension of the functions in reach), the degree profile `W_k` (share of the weight on monomials of `k` angles, the Pauli weight of `O_k`) and its mean, the share `A_k` of the degree-`k` functions in reach, the effective dimension `d_eff` of the features for angles uniform on `[0, pi]`, and the weight on monomials with an odd number of `sin` (zero for any dynamics that keeps `prod_i Z_i` or `sum_i Z_i`).

Four dynamics: Haar U(4), U(1), Z2 parity and Clifford brickworks, depth `T = N`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here or through the environment variable of the same name. The default grid (`N = 4, 6, 8` with 10 circuits each, `N = 10` with 2, four dynamics, 21 values of `p`) takes about 1.5 hours with `MAX_JOBS = 4`; the `N = 10` runs (about 40 min each on one core) are most of it. For a quick look, drop `10:2` from `SIZES`. `RUN = False` reads back the latest sweep of `outputs/fourier_expressivity/`; a sweep stopped midway is completed with `SKIP_DONE = 1` and `OUT_DIR` set to its folder.

In [ ]:
MODELS = os.environ.get("MODELS", "haar u1 z2 clifford")
SIZES = os.environ.get("SIZES", "4:10 6:10 8:10 10:2")  # N:number of circuits
T_FACTOR = os.environ.get("T_FACTOR", "1")  # depth T = T_FACTOR * N
P_VALUES = os.environ.get("P_VALUES", " ".join(f"{0.05 * k:.2f}" for k in range(21)))
MAX_JOBS = os.environ.get("MAX_JOBS", "4")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
OUT_DIR = os.environ.get("OUT_DIR", "")  # empty: a new sweep folder
SKIP_DONE = os.environ.get("SKIP_DONE", "0")
RUN = os.environ.get("RUN", "1") == "1"

## Launch the sweep

One run per dynamics, size and circuit (`experiments/fourier_expressivity/fourier_expressivity.sh`); each run computes `C(p)` for every `p` and writes `metrics.json`.

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

EXPERIMENT = "fourier_expressivity"
n_runs = len(MODELS.split()) * sum(int(size.split(":")[1]) for size in SIZES.split())
if RUN:
    sweep_dir = Path(OUT_DIR) if OUT_DIR else new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "MODELS": MODELS,
        "SIZES": SIZES,
        "T_FACTOR": T_FACTOR,
        "P_VALUES": P_VALUES,
        "MAX_JOBS": MAX_JOBS,
        "SKIP_DONE": SKIP_DONE,
        "OUT_DIR": str(sweep_dir),
    }
    run_with_progress(
        ["bash", "experiments/fourier_expressivity/fourier_expressivity.sh"],
        total=n_runs,
        env=env,
        show=("model", "N", "seed"),
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back and summarize

Averages over the circuits, per dynamics and size. The table gives, at a few `p`: the rank of `C(p)` over the number of features `T N + N`, the rank of the final `Z_i` alone, the mean degree of the final `Z_i` over `N`, `d_eff`, and the weight on the odd-`sin` monomials.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "plot_fourier", "experiments/fourier_expressivity/plot_fourier_expressivity.py"
)
plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot)

runs = plot.load_runs(sweep_dir)
summary = plot.summarize(runs)
rows = plot.table(summary)
print(f"{len(runs)} runs")
print(
    f"{'model':>9} {'N':>3} {'p':>5} {'rank/K':>7} {'rank_fin':>8}"
    f" {'k_fin/N':>8} {'d_eff':>6} {'odd_x':>7}"
)
for r in rows:
    print(
        f"{r['model']:>9} {r['N']:>3} {r['p']:5.2f} {r['rank/features']:7.3f}"
        f" {r['final_rank']:8.1f} {r['k_final/N']:8.3f} {r['d_eff']:6.2f}"
        f" {r['odd_x']:7.1e}"
    )

## Figures

Saved in the sweep folder, each with its data:

1. `accessible_rank`: rank of `C(p)` over the number of features. How many independent functions of the input the record and the final readout give.
2. `final_mean_degree`: mean degree over `N` of the final `Z_i`. The degree is the number of input angles a coefficient couples: in the volume-law phase the Heisenberg operators spread and it should stay high, in the area-law phase the measurements should keep it low. A crossing in `N` would point to `p_c`.
3. `record_mean_degree`: the same over all features, record included.
4. `effective_dimension`: participation ratio of the feature covariance, angles uniform on `[0, pi]`.
5. `odd_x_weight`: the selection rule; zero to machine precision for U(1) and Z2.
6. `degree_profile_N<max>`: `W_k` of the final `Z_i` over `(p, k)` at the largest `N`.
7. `degree_by_time_N<max>`: mean degree of the record row `M_t` over `(p, t)`: the light cone of each measurement time.
8. `accessibility_by_degree_N<max>`: `A_k`, the share of the degree-`k` functions in reach, for `p = 0, 0.25, 0.5, 1`.

In [ ]:
figure_paths = plot.draw_all(sweep_dir, show=True)
figure_path = figure_paths["final_mean_degree"]
for name, path in figure_paths.items():
    print(f"{name}: {path}")